In [5]:
// 1. deps + structs
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"

use std::collections::{BTreeMap, BTreeSet};
use std::process::Command;

use serde::{Deserialize, Serialize};

#[derive(Debug, Clone, Serialize, Deserialize)]
struct CommentSourceRecord {
    #[serde(rename = "postId")]
    post_id: u64,
    id: u64,
    name: String,
    email: String,
    body: String,
}

In [6]:
// 2. fetch data with curl
let output = Command::new("curl")
    .arg("-s")
    .arg("https://jsonplaceholder.typicode.com/comments")
    .output()?;

if !output.status.success() {
    panic!("curl failed with status: {:?}", output.status.code());
}

let content = String::from_utf8(output.stdout)?;
println!("bytes: {}", content.len());
println!("{}", &content[..content.len().min(300)]);


bytes: 157745
[
  {
    "postId": 1,
    "id": 1,
    "name": "id labore ex et quam laborum",
    "email": "Eliseo@gardner.biz",
    "body": "laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium"
  },
  {
    "postI


In [7]:
// 3. parse json
let records: Vec<CommentSourceRecord> = serde_json::from_str(&content)?;

println!("row_count: {}", records.len());
println!("first_row: {:#?}", records.get(0));


row_count: 500
first_row: Some(
    CommentSourceRecord {
        post_id: 1,
        id: 1,
        name: "id labore ex et quam laborum",
        email: "Eliseo@gardner.biz",
        body: "laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium",
    },
)


In [8]:
// 3. profile source
let unique_ids: BTreeSet<_> = records.iter().map(|r| r.id).collect();
let unique_post_ids: BTreeSet<_> = records.iter().map(|r| r.post_id).collect();

let blank_names = records.iter().filter(|r| r.name.trim().is_empty()).count();
let blank_emails = records.iter().filter(|r| r.email.trim().is_empty()).count();
let blank_bodies = records.iter().filter(|r| r.body.trim().is_empty()).count();

let min_body_len = records.iter().map(|r| r.body.len()).min().unwrap_or(0);
let max_body_len = records.iter().map(|r| r.body.len()).max().unwrap_or(0);

println!("unique_ids       : {}", unique_ids.len());
println!("unique_post_ids  : {}", unique_post_ids.len());
println!("blank_names      : {}", blank_names);
println!("blank_emails     : {}", blank_emails);
println!("blank_bodies     : {}", blank_bodies);
println!("min_body_len     : {}", min_body_len);
println!("max_body_len     : {}", max_body_len);


unique_ids       : 500
unique_post_ids  : 100
blank_names      : 0
blank_emails     : 0
blank_bodies     : 0
min_body_len     : 78
max_body_len     : 263


In [9]:
// 4. simple distributions
let mut comments_per_post: BTreeMap<u64, usize> = BTreeMap::new();
for record in &records {
    *comments_per_post.entry(record.post_id).or_insert(0) += 1;
}

println!("comments per post_id (first 10):");
for (post_id, count) in comments_per_post.iter().take(10) {
    println!("  post_id={} -> {}", post_id, count);
}


comments per post_id (first 10):
  post_id=1 -> 5
  post_id=2 -> 5
  post_id=3 -> 5
  post_id=4 -> 5
  post_id=5 -> 5
  post_id=6 -> 5
  post_id=7 -> 5
  post_id=8 -> 5
  post_id=9 -> 5
  post_id=10 -> 5


()

In [10]:
// 5. candidate contract checks
assert!(!records.is_empty(), "source payload is empty");
assert!(records.iter().all(|r| !r.name.trim().is_empty()), "found blank name");
assert!(records.iter().all(|r| !r.email.trim().is_empty()), "found blank email");
assert!(records.iter().all(|r| !r.body.trim().is_empty()), "found blank body");

println!("basic source checks passed");


basic source checks passed


## Findings

- payload is a JSON array
- `id` appears to be the natural primary/business key
- `post_id` looks like a parent/reference key
- `name`, `email`, and `body` should be required in bronze
- `email` should likely be normalized in silver
- `body_length` may be a useful derived silver field
- bronze source should probably rename `id` to `comment_id`
